# 02 — Feature Engineering

Demonstrates and validates the three feature groups:
1. **Dietary flags** — boolean suitability labels (diabetic, gluten-free, high-protein, etc.)
2. **Numeric features** — engineered ratios and normalized values
3. **Composite text** — fused text field for embedding

In [ ]:
import os
os.chdir('..')  # run from project root so all data/ paths resolve correctly

import sys
sys.path.insert(0, '.')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

In [ ]:
featured = pd.read_parquet('data/processed/recipes_featured.parquet')
print(f'{len(featured):,} recipes, {len(featured.columns)} columns')
print(featured.columns.tolist())

## 1. Dietary Flags Validation

Verify that flag thresholds are correctly applied.

In [ ]:
# Verify diabetic_friendly: glycemic_proxy < 8
diabetic = featured[featured['is_diabetic_friendly']]
non_diabetic = featured[~featured['is_diabetic_friendly']]

print('Diabetic-friendly recipes:')
print(f'  Count: {len(diabetic):,}')
print(f'  Avg sugar_pdv: {diabetic.sugar_pdv.mean():.1f}% (non-diabetic: {non_diabetic.sugar_pdv.mean():.1f}%)')
print(f'  Avg carbs_pdv: {diabetic.carbohydrates_pdv.mean():.1f}% (non-diabetic: {non_diabetic.carbohydrates_pdv.mean():.1f}%)')

# Verify gluten_free: ingredient-level check
print()
gf = featured[featured['is_gluten_free']]
print(f'Gluten-free recipes: {len(gf):,}')
# Spot check — should not contain wheat/flour
gluten_in_gf = gf['ingredients_str'].str.contains('flour|wheat|barley', case=False).sum()
print(f'  Recipes incorrectly containing gluten keywords: {gluten_in_gf}')

## 2. Nutritional Correlation Heatmap

In [ ]:
num_cols = ['calories', 'total_fat_pdv', 'sugar_pdv', 'sodium_pdv',
            'protein_pdv', 'carbohydrates_pdv', 'protein_to_calorie_ratio',
            'sugar_to_carb_ratio', 'complexity_score']

corr = featured[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Composite Text Example

Verify what gets sent to the embedding model.

In [ ]:
# Show example recipe_text for a diabetic-friendly high-protein recipe
sample = featured[featured['is_diabetic_friendly'] & featured['is_high_protein']].iloc[0]
print(f'Recipe: {sample["name"].title()}')
print(f'Dietary flags: diabetic_friendly={sample.is_diabetic_friendly}, high_protein={sample.is_high_protein}')
print()
print('recipe_text (sent to embedding model):')
print('-' * 60)
print(sample['recipe_text'][:600] + '...')
print(f'\nTotal length: {len(sample["recipe_text"])} chars')

## 4. Profile Overlap Analysis

Which profiles commonly co-occur? (useful for query design)

In [ ]:
flag_cols = [c for c in featured.columns if c.startswith('is_')]
flag_df = featured[flag_cols].astype(int)
overlap = flag_df.T.dot(flag_df)

plt.figure(figsize=(10, 8))
labels = [c.replace('is_', '').replace('_', '\n') for c in flag_cols]
sns.heatmap(overlap, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Dietary Profile Overlap (recipe counts)', fontsize=14)
plt.tight_layout()
plt.show()